In [1]:
!pip install pydub
!pip install imageio-ffmpeg

In [2]:
!pip install python-dotenv

In [2]:
import json
import os
import time
from pydub import AudioSegment
import imageio_ffmpeg
from dotenv import load_dotenv

# ----------------------------------------------------
# 1. 환경 및 경로 세팅 (C드라이브 전용 테스트)
# ----------------------------------------------------
AudioSegment.converter = imageio_ffmpeg.get_ffmpeg_exe()
load_dotenv()

INPUT_DIR = os.getenv("AIHUB_INPUT_DIR")
OUTPUT_BASE_DIR = os.getenv("AIHUB_OUTPUT_DIR")

if not INPUT_DIR or not OUTPUT_BASE_DIR:
    raise ValueError(".env 파일 경로를 확인해주세요!")

# ⭐️ 오직 '라디오/경제' 폴더만 타겟팅합니다!
LABELING_BASE_DIR = os.path.join(INPUT_DIR, "labeling")
TARGET_LABELING_DIR = os.path.join(LABELING_BASE_DIR, "라디오", "경제")
RESOURCE_DIR = os.path.join(INPUT_DIR, "resource")
OUTPUT_DIR = os.path.join(OUTPUT_BASE_DIR, "dataset_processed")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------------------------------
# 2. 핵심 썰기 함수 (0.1초 휴식 포함)
# ----------------------------------------------------
def process_single_file(json_path, audio_path, output_dir, base_filename):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    audio = AudioSegment.from_file(audio_path)
    utterances = data['utterance']
    
    current_chunk_text = ""
    chunk_start_time = float(utterances[0]['start'])
    last_end_time = chunk_start_time
    chunk_id = 1
    
    for u in utterances:
        u_start = float(u['start'])
        u_end = float(u['end'])
        
        if (u_end - chunk_start_time) > 28.0 and current_chunk_text != "":
            audio_chunk = audio[int(chunk_start_time * 1000):int(last_end_time * 1000)]
            
            # C드라이브 저장
            audio_chunk.export(os.path.join(output_dir, f"{base_filename}_chunk_{chunk_id}.wav"), format="wav")
            with open(os.path.join(output_dir, f"{base_filename}_chunk_{chunk_id}.txt"), "w", encoding="utf-8") as tf:
                tf.write(current_chunk_text.strip())
            time.sleep(0.1) 
                
            chunk_start_time = u_start
            current_chunk_text = ""
            chunk_id += 1

        current_chunk_text += u['form'] + " "
        last_end_time = u_end

    if current_chunk_text != "":
        audio_chunk = audio[int(chunk_start_time * 1000):int(last_end_time * 1000)]
        audio_chunk.export(os.path.join(output_dir, f"{base_filename}_chunk_{chunk_id}.wav"), format="wav")
        with open(os.path.join(output_dir, f"{base_filename}_chunk_{chunk_id}.txt"), "w", encoding="utf-8") as tf:
            tf.write(current_chunk_text.strip())
        time.sleep(0.1)

# ----------------------------------------------------
# 3. 타겟팅 루프 실행
# ----------------------------------------------------
print(f"🚀 [C드라이브 격리 테스트] '{TARGET_LABELING_DIR}' 작업 시작...\n")
processed_count = 0
skipped_count = 0
error_count = 0

for root, dirs, files in os.walk(TARGET_LABELING_DIR):
    for file in files:
        if file.endswith(".json"):
            json_path = os.path.join(root, file)
            
            # resource 폴더 내의 정확한 짝꿍 wav 경로 찾기
            relative_path = os.path.relpath(root, LABELING_BASE_DIR)
            wav_dir = os.path.join(RESOURCE_DIR, relative_path)
            wav_filename = file.replace(".json", ".wav")
            wav_path = os.path.join(wav_dir, wav_filename)
            
            if os.path.exists(wav_path):
                base_name = file.replace(".json", "")
                
                if os.path.exists(os.path.join(OUTPUT_DIR, f"{base_name}_chunk_1.wav")):
                    skipped_count += 1
                    if skipped_count % 100 == 0: # 파일이 적을 테니 100개 단위로 로그
                        print(f"⏭️ {skipped_count}개 파일 스킵 완료...")
                    continue
                
                print(f"🔪 [{processed_count + 1}번째] {base_name} 썰기 중...")
                
                try:
                    process_single_file(json_path, wav_path, OUTPUT_DIR, base_name)
                    processed_count += 1
                except Exception as e:
                    print(f"  🔴 [에러] {base_name} 실패 (사유: {e})")
                    error_count += 1
            else:
                print(f"  ⚠️ [경고] 원본 오디오 없음: {wav_path}")

print(f"\n🎉 [C드라이브 테스트 완료] 성공: {processed_count}개 | 스킵: {skipped_count}개 | 에러: {error_count}개")

🚀 [C드라이브 격리 테스트] 'C:\Users\최재현\Desktop\AI_Whisper_Tuning\labeling\라디오\경제\labeling\라디오\경제' 작업 시작...


🎉 [C드라이브 테스트 완료] 성공: 0개 | 스킵: 0개 | 에러: 0개
